In [1]:
import pandas as pd
import numpy as np
import glob


In [2]:
# all 12 files share the exact same 17 columns, in the same order,
# just with inconsistent spacing/typos in the header text -- so we
# rename by position instead of trying to match messy header strings

col_names = [
    "datetime", "choke", "thp", "tht", "p_ds_choke", "t_ds_choke",
    "chp", "cht", "tap", "bcp", "bottom_press", "bottom_temp",
    "water_pct", "emulsion_pct", "sand_pct", "bsw_pct", "h2s_ppm",
]

files = glob.glob("*dataset.csv")
files


['wel 2  long string GL dataset.csv',
 'well 1 dataset.csv',
 'well 10 dataset.csv',
 'Well 11 dataset.csv',
 'well 12 dataset.csv',
 'well 13 dataset.csv',
 'well 14 dataset.csv',
 'well 3  GL  dataset.csv',
 'well 4  dataset.csv',
 'well 7 dataset.csv',
 'well 8 dataset.csv',
 'well 9 dataset.csv']

In [3]:
dfs = []

for f in files:
    d = pd.read_csv(f)
    d = d.loc[:, ~d.columns.str.contains("Unnamed")]   # drop stray blank columns
    d.columns = col_names
    d["well_id"] = f.replace(".csv", "")
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
df.shape


(12937, 18)

In [4]:
# checking empty values
df.isnull().sum()


datetime         1415
choke            2133
thp              2133
tht             11029
p_ds_choke       2137
t_ds_choke       2137
chp              2133
cht             12937
tap              2133
bcp              2133
bottom_press    12057
bottom_temp     12057
water_pct        2180
emulsion_pct     2181
sand_pct         2182
bsw_pct          2182
h2s_ppm          2138
well_id             0
dtype: int64

In [5]:
# clean up dtypes and bad rows
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df = df.dropna(subset=["datetime"])          # drops blank filler rows in some files
df = df.sort_values(["well_id", "datetime"]).reset_index(drop=True)

num_cols = [c for c in col_names if c != "datetime"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

df.shape


(11522, 18)

In [6]:
# feature extraction

df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month

# lag + rate of change, computed per well so wells don't bleed into each other
for c in num_cols:
    df[f"{c}_lag1"] = df.groupby("well_id")[c].shift(1)
    df[f"{c}_delta1"] = df[c] - df[f"{c}_lag1"]

# 24h rolling mean/std (12 samples at ~2h sampling)
for c in num_cols:
    roll = df.groupby("well_id")[c].rolling(window=12, min_periods=3)
    df[f"{c}_roll_mean24h"] = roll.mean().reset_index(level=0, drop=True)
    df[f"{c}_roll_std24h"] = roll.std().reset_index(level=0, drop=True)

# domain features
df["press_drawdown"] = df["bottom_press"] - df["thp"]
df["temp_gradient"] = df["bottom_temp"] - df["tht"]
df["choke_press_drop"] = df["thp"] - df["p_ds_choke"]

df.shape


(11522, 88)

In [7]:
# split into features / target
# (pick whichever column you actually want to predict -- water_pct here as an example)

x = df.drop(columns=["water_pct", "datetime", "well_id"])
y = df["water_pct"]

x.shape, y.shape


((11522, 85), (11522,))

In [8]:
df.to_csv("all_wells_features.csv", index=False)
